In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import os
from datetime import UTC, datetime

import cartopy.crs as ccrs
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import xvec

import matplotlib
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.colors import BivarColormapFromImage

from dotenv import find_dotenv, load_dotenv

from utils import analysis_utils, isku_utils, plotting_utils, stats_utils

importlib.reload(analysis_utils)
importlib.reload(isku_utils)
importlib.reload(plotting_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'utils.plotting_utils' from '/home/emily_zuetell/projects/poreallas/utils/plotting_utils.py'>

In [2]:
load_dotenv(
    find_dotenv("october_local.env", raise_error_if_not_found=True), override=True
)

EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]

# Climate Data
ERA5_URI = os.environ["POREALLAS_ERA5_URI"]

In [3]:
# Baseline period for Impact
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)
BASELINE_PERIOD = analysis_utils.get_baseline_period(effect, years=30)
# Define Forecast Months
FC_MONTHS = [10, 11, 12, 1, 2, 3]

config = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=True,
    age_weight=True,
    months=FC_MONTHS,
    hotonly="hotonly",
    dims=["number", "sample"],
)

In [ ]:
def _write_zarr(ds, path, attrs):
    ds = ds.reset_coords(drop=True)
    ds.attrs = {"version": config.version, "fc_months": list(config.months), **attrs}
    ds.drop_encoding().chunk(-1).to_zarr(path, mode="w", consolidated=True)


def _impact(hotonly, ensemble):
    # Compute Impact and standardize selecting 6-month period
    return analysis_utils.compute_impact(
        effect.chunk({dim: -1 for dim in config.dims}), config, ensemble=ensemble, hotonly=hotonly
    ).sel(month=config.months)


def _map_stats(hotonly):
    # Build bivariate map stats
    ds_6 = _impact(hotonly, ensemble=True).sum(dim="month")
    return xr.Dataset({
        "mean": ds_6.mean(dim=["sample", "number"]),
        "std": ds_6.std(dim=["sample", "number"]),
    })

## Panel 2a/b: Bivariate Impact Maps

In [6]:
def save_panel_a(path):
    _write_zarr(_map_stats("hotonly"), path, {"hazard": "hotonly", "units": "deaths/100k, 6-month total"})


def save_panel_b(path):
    _write_zarr(_map_stats("coldonly"), path, {"hazard": "coldonly", "units": "deaths/100k, 6-month total"})

In [7]:
save_panel_a(f"../data/figures/{config.version}_fig2a.zarr")
save_panel_b(f"../data/figures/{config.version}_fig2b.zarr")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=35, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/home/emily_zuetell/projects/poreallas/

## Panel 2c: Monthly Global Time Series

In [ ]:
def save_panel_c(path):
    series = []
    for hotonly in ["hotonly", "coldonly"]:
        impact = _impact(hotonly, ensemble=True) # Compute hot and cold-only impact, BY MONTH, including uncertainty from ensemble members and samples
        w = config.socioeconomics.pop.sel(region=impact.region).fillna(0) # Pop-Weighting
        series.append(impact.weighted(w).mean(dim="region"))
    da = xr.concat(series, dim=pd.Index(["hotonly", "coldonly"], name="hazard"))
    _write_zarr(da.to_dataset(name="impact"), path, {"weights": "pop", "units": "deaths/100k per month"})

In [9]:
save_panel_c(f"../data/figures/{config.version}_fig2c.zarr")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/home/emily_zuetell/projects/poreallas/.

## Panel 2d: Box Plot - Income Group

In [10]:
def save_panel_d(path):
    vals = {}
    for hotonly in ["hotonly", "coldonly"]:
        impact = _impact(hotonly, ensemble=False).sum(dim="month") # 6-month Cumulative hotonly impact
        extra = [d for d in ["number", "sample"] if d in impact.dims] # Check for additional dims
        vals[f"impact_{hotonly}"] = impact.mean(dim=extra) if extra else impact # Use mean vals
    aligned = xr.align(
        *vals.values(), config.socioeconomics.gdppc, config.socioeconomics.pop, join="inner" # Add socioeconomics (for income group) and pop (for equal-pop quartiles, weighted median)
    )
    ds = xr.Dataset(dict(zip([*vals, "gdppc", "pop"], aligned)))
    _write_zarr(ds, path, {"ensemble": False, "units": "deaths/100k, 6-month total"})

In [11]:
save_panel_d(f"../data/figures/{config.version}_fig2d.zarr")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=35, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/home/emily_zuetell/projects/poreallas/